In [1]:
# Cell 1 - Imports
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path
import json

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ GPU available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

✅ PyTorch version: 2.11.0+cpu
✅ GPU available: False
✅ Using device: cpu


In [2]:
# Cell 2 - Load dataset info
BASE_DIR = Path("..").resolve()

# Load class names we saved earlier
with open(BASE_DIR / "utils" / "dataset_info.json", "r") as f:
    dataset_info = json.load(f)

NUM_CLASSES = dataset_info["num_classes"]
CLASS_NAMES = dataset_info["class_names"]

print(f"✅ Dataset info loaded!")
print(f"  Number of classes: {NUM_CLASSES}")
print(f"  Classes: {CLASS_NAMES}")

✅ Dataset info loaded!
  Number of classes: 20
  Classes: ['cat_flea_allergy', 'cat_healthy', 'cat_ringworm', 'cat_scabies', 'dog_demodicosis', 'dog_dermatitis', 'dog_fungal_infections', 'dog_healthy', 'dog_hypersensitivity', 'dog_ringworm', 'human_atopic_dermatitis', 'human_basal_cell_carcinoma', 'human_benign_keratosis', 'human_eczema', 'human_melanocytic_nevi', 'human_melanoma', 'human_psoriasis', 'human_seborrheic_keratoses', 'human_tinea_ringworm', 'human_warts_molluscum']


In [3]:
# Cell 3 - Build EfficientNet Model
class SkinDiseaseModel(nn.Module):
    def __init__(self, num_classes):
        super(SkinDiseaseModel, self).__init__()
        
        # Load pretrained EfficientNet
        self.backbone = models.efficientnet_b0(weights='DEFAULT')
        
        # Freeze early layers
        for param in list(self.backbone.parameters())[:-20]:
            param.requires_grad = False
        
        # Get number of features
        num_features = self.backbone.classifier[1].in_features
        
        # Remove original classifier
        self.backbone.classifier = nn.Identity()
        
        # Disease classification head
        self.disease_head = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
        
        # Severity head (0=mild, 1=moderate, 2=severe)
        self.severity_head = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 3)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        disease_out  = self.disease_head(features)
        severity_out = self.severity_head(features)
        return disease_out, severity_out

# Create model
model = SkinDiseaseModel(num_classes=NUM_CLASSES).to(device)

print("✅ Model created successfully!")
print(f"\n📊 Model Summary:")
print(f"  Backbone:       EfficientNet-B0")
print(f"  Disease heads:  {NUM_CLASSES} classes")
print(f"  Severity heads: 3 classes (mild/moderate/severe)")
print(f"  Device:         {device}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\shruti/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:01<00:00, 15.8MB/s]

✅ Model created successfully!

📊 Model Summary:
  Backbone:       EfficientNet-B0
  Disease heads:  20 classes
  Severity heads: 3 classes (mild/moderate/severe)
  Device:         cpu


In [4]:
# Cell 4 - Test model with dummy input
# This checks model works correctly before training

dummy_input = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    disease_out, severity_out = model(dummy_input)

print("✅ Model forward pass successful!")
print(f"\n📊 Output shapes:")
print(f"  Disease output:  {disease_out.shape}")
print(f"  Severity output: {severity_out.shape}")
print(f"\n📋 Expected:")
print(f"  Disease:  torch.Size([1, {NUM_CLASSES}])")
print(f"  Severity: torch.Size([1, 3])")

✅ Model forward pass successful!

📊 Output shapes:
  Disease output:  torch.Size([1, 20])
  Severity output: torch.Size([1, 3])

📋 Expected:
  Disease:  torch.Size([1, 20])
  Severity: torch.Size([1, 3])


In [5]:
# Cell 5 - Count model parameters
def count_parameters(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = total - trainable
    
    print(f"📊 Model Parameters:")
    print(f"  Total:     {total:,}")
    print(f"  Trainable: {trainable:,}")
    print(f"  Frozen:    {frozen:,}")
    print(f"\n💡 Only trainable parameters will be updated during training")

count_parameters(model)

📊 Model Parameters:
  Total:     4,504,979
  Trainable: 1,627,207
  Frozen:    2,877,772

💡 Only trainable parameters will be updated during training


In [7]:
# Cell 6 - Save model architecture
import os

# Save model architecture to utils folder
save_path = BASE_DIR / "model" / "model.py"

model_code = '''import torch
import torch.nn as nn
from torchvision import models

class SkinDiseaseModel(nn.Module):
    def __init__(self, num_classes):
        super(SkinDiseaseModel, self).__init__()
        
        # Load pretrained EfficientNet
        self.backbone = models.efficientnet_b0(weights="DEFAULT")
        
        # Freeze early layers
        for param in list(self.backbone.parameters())[:-20]:
            param.requires_grad = False
        
        # Get number of features
        num_features = self.backbone.classifier[1].in_features
        
        # Remove original classifier
        self.backbone.classifier = nn.Identity()
        
        # Disease classification head
        self.disease_head = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
        
        # Severity head
        self.severity_head = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 3)
        )
    
    def forward(self, x):
        features     = self.backbone(x)
        disease_out  = self.disease_head(features)
        severity_out = self.severity_head(features)
        return disease_out, severity_out
'''

with open(save_path, "w") as f:
    f.write(model_code)

print(f"✅ Model architecture saved to {save_path}")
print("\n🎉 Model Notebook Complete!")


✅ Model architecture saved to C:\Users\shruti\Desktop\project\Skin-Disease Predictor\model\model.py

🎉 Model Notebook Complete!
